<a href="https://colab.research.google.com/github/peterdomjan-maker/Biomed2026/blob/main/S7_PYG_STUDENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BIOMED-DATA 26 · Session 7
## A graph neural network you can train (PyTorch Geometric)
**Build a patient-similarity graph and train a 2-layer GCN to predict the diagnosis — the classic node-classification recipe**

> **STUDENT — fill in the blanks**
>
> Dataset: UCI Heart Disease (`ucimlrepo`, id=45). Runs in Colab: `Runtime → Run all`.
> Semmelweis University · Luca Szegletes (BME)
---

This is the practical companion to the message-passing lecture. Instead of coding message passing by hand,
we use **PyTorch Geometric (PyG)** and its `GCNConv` layer — the standard tool for graph neural networks.

The recipe is the classic **node classification** setup:
1. each **patient is a node**, with their standardised measurements as features;
2. connect each patient to their most similar others (a k-NN graph) — these are the **edges**;
3. a 2-layer **GCN** passes messages along the edges and predicts each node's label;
4. we train on some nodes and test on the rest (a transductive split).

It is a small graph, so it trains in seconds on CPU. On independent patients with an *invented* similarity graph,
expect it to land near a plain logistic baseline — the honest result the lecture warned about.

In [1]:
# PyTorch is preinstalled in Colab. Install PyTorch Geometric (CPU is fine for this small graph).
!pip -q install torch_geometric ucimlrepo scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 12.7 MB/s eta 0:00:00


> **If PyG install complains:** `pip install torch_geometric` alone runs `GCNConv` on CPU in current Colab.
> If you hit a scatter/sparse error, also run `pip install pyg-lib torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-$(python -c 'import torch;print(torch.__version__)').html` — but for this small graph it is usually not needed.

## 0. Node features: one patient per node

In [2]:
import numpy as np, torch, torch.nn.functional as F
from ucimlrepo import fetch_ucirepo
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import kneighbors_graph
from sklearn.metrics import roc_auc_score
torch.manual_seed(0); np.random.seed(0)

heart = fetch_ucirepo(id=45)
X = heart.data.features.copy()
y = (heart.data.targets["num"] > 0).astype(int).values
num = ["age","trestbps","chol","thalach","oldpeak"]
# node features: standardised numeric measurements
Xn = StandardScaler().fit_transform(SimpleImputer(strategy="median").fit_transform(X[num])).astype("float32")
print("Nodes (patients):", Xn.shape[0], "| features per node:", Xn.shape[1])

Nodes (patients): 303 | features per node: 5


## 1. Build the graph (edges = k nearest patients) and the train/test masks

In [3]:
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected

# 1) EDGES: connect each patient to its k nearest neighbours
A = kneighbors_graph(Xn, n_neighbors=8, mode="connectivity", include_self=False).tocoo()
edge_index = torch.tensor(np.vstack([A.row, A.col]), dtype=torch.long)
edge_index = to_undirected(edge_index)

x = torch.tensor(Xn)
labels = torch.tensor(y, dtype=torch.long)

# 3) train / test masks (70% train)
n = x.size(0); perm = np.random.default_rng(0).permutation(n)
train_mask = torch.zeros(n, dtype=torch.bool); test_mask = torch.zeros(n, dtype=torch.bool)
train_mask[perm[:int(0.7*n)]] = True
test_mask[perm[int(0.7*n):]] = True

# TODO: build the graph Data object from x, edge_index, labels and the two masks
data = Data(x=x, edge_index=edge_index, y=labels, train_mask=train_mask, test_mask=test_mask)
print(data)

Data(x=[303, 5], edge_index=[2, 3364], y=[303], train_mask=[303], test_mask=[303])


## 2. The model: a 2-layer GCN

In [4]:
from torch_geometric.nn import GCNConv

class GCN(torch.nn.Module):
    def __init__(self, in_ch, hidden, out_ch):
        super().__init__()
        # TODO: two GCNConv layers: in_ch -> hidden, then hidden -> out_ch
        self.conv1 = GCNConv(in_ch, hidden)
        self.conv2 = GCNConv(hidden, out_ch)
    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x

model = GCN(in_ch=data.num_features, hidden=16, out_ch=2)
print(model)

GCN(
  (conv1): GCNConv(5, 16)
  (conv2): GCNConv(16, 2)
)


## 3. Train it (loss on train nodes, evaluate on test nodes)

In [5]:
opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

def evaluate(mask):
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        pred = out.argmax(dim=1)
        acc = (pred[mask] == data.y[mask]).float().mean().item()
        prob = F.softmax(out, dim=1)[:, 1].numpy()
        auc = roc_auc_score(data.y[mask].numpy(), prob[mask.numpy()])
    return acc, auc

for epoch in range(100):
    model.train(); opt.zero_grad()
    out = model(data.x, data.edge_index)
    # TODO: cross-entropy loss on the TRAIN nodes only (use data.train_mask)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward(); opt.step()
    if (epoch + 1) % 20 == 0:
        acc, auc = evaluate(data.test_mask)
        print(f"epoch {epoch+1:3d}  loss {loss.item():.3f}  test acc {acc:.3f}  test AUC {auc:.3f}")

acc, auc = evaluate(data.test_mask)
print(f"\nFinal — test accuracy {acc:.3f}, test AUC {auc:.3f}")

epoch  20  loss 0.517  test acc 0.714  test AUC 0.783
epoch  40  loss 0.494  test acc 0.692  test AUC 0.782
epoch  60  loss 0.506  test acc 0.703  test AUC 0.780
epoch  80  loss 0.486  test acc 0.703  test AUC 0.777
epoch 100  loss 0.483  test acc 0.681  test AUC 0.775

Final — test accuracy 0.681, test AUC 0.775


## 4. The honest baseline (no graph)

In [6]:
# Honest baseline: logistic regression on the SAME features, ignoring the graph
from sklearn.linear_model import LogisticRegression
tr = data.train_mask.numpy(); te = data.test_mask.numpy()
lr = LogisticRegression(max_iter=1000).fit(Xn[tr], y[tr])
print("GCN (uses the graph)   test AUC:", round(evaluate(data.test_mask)[1], 3))
print("Logistic (no graph)    test AUC:", round(roc_auc_score(y[te], lr.predict_proba(Xn[te])[:,1]), 3))

GCN (uses the graph)   test AUC: 0.775
Logistic (no graph)    test AUC: 0.757


## Questions to think about
1. Did the GCN beat the graph-free logistic baseline? On independent patients with an invented similarity graph, why is the gain usually small?
2. Change `n_neighbors` (the graph) or `hidden` (the model). Does a denser graph or a bigger model help — or does it over-smooth / overfit?
3. Where would a GNN genuinely shine instead: a real molecule graph, a referral network, or a contact-tracing graph?

*This is the practical companion to the message-passing lecture. For the idea coded by hand, see `S7_DEMO.ipynb`.*